In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("/Users/inayasiddiqui/Documents/Math6243FinalProject/Biodiversity-Predictors/data/processed/marine_data_cleaned.csv")

print(df.head())
print(df.shape)

# target variables
y_richness = df["species_richness"]
y_log_density = np.log1p(df["species_richness"]) - df["log_area"]

df["log_density"] = np.log1p(df["species_richness"]) - df["log_area"]

print(df[["species_richness", "log_area", "log_density"]].head())

# separate features
X_rich = df.drop(columns=["species_richness", "STUDY_ID"], errors="ignore")
X_density = df.drop(columns=["species_richness", "STUDY_ID", "log_area"], errors="ignore")

def prepare_features(X):
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

    X_cat = pd.get_dummies(X[cat_cols], drop_first=True)
    X_final = pd.concat([X[num_cols], X_cat], axis=1)

    return X_final, num_cols, cat_cols

# prepare both feature matrices
X_rich_final, num_cols_rich, cat_cols_rich = prepare_features(X_rich)
X_density_final, num_cols_density, cat_cols_density = prepare_features(X_density)

print("\nRichness feature matrix shape:", X_rich_final.shape)
print("Density feature matrix shape:", X_density_final.shape)


indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# richness split
X_rich_train = X_rich_final.iloc[train_idx].copy()
X_rich_test = X_rich_final.iloc[test_idx].copy()
y_rich_train = y_richness.iloc[train_idx]
y_rich_test = y_richness.iloc[test_idx]

# density split
X_density_train = X_density_final.iloc[train_idx].copy()
X_density_test = X_density_final.iloc[test_idx].copy()
y_density_train = y_log_density.iloc[train_idx]
y_density_test = y_log_density.iloc[test_idx]

# Scale numeric columns
scaler_rich = StandardScaler()
X_rich_train[num_cols_rich] = scaler_rich.fit_transform(X_rich_train[num_cols_rich])
X_rich_test[num_cols_rich] = scaler_rich.transform(X_rich_test[num_cols_rich])

scaler_density = StandardScaler()
X_density_train[num_cols_density] = scaler_density.fit_transform(X_density_train[num_cols_density])
X_density_test[num_cols_density] = scaler_density.transform(X_density_test[num_cols_density])

# optional: convert to arrays for sklearn models
X_rich_train_scaled = X_rich_train.values
X_rich_test_scaled = X_rich_test.values

X_density_train_scaled = X_density_train.values
X_density_test_scaled = X_density_test.values

print("\nRichness training set:", X_rich_train_scaled.shape)
print("Richness test set:", X_rich_test_scaled.shape)
print("Density training set:", X_density_train_scaled.shape)
print("Density test set:", X_density_test_scaled.shape)

   STUDY_ID  LATITUDE  LONGITUDE  species_richness  total_abundance  \
0       164  37.81020 -122.32300                13            126.0   
1       245  50.85045    3.61936                 9             50.0   
2       245  50.88528    3.69284                 6             49.0   
3       245  50.91131    3.41330                15           1106.0   
4       245  50.91145    3.41317                15            420.0   

   mean_depth  n_samples  n_years  year_start  year_end  ... temp_range_proxy  \
0    1.000000          3        2        2000      2001  ...        11.343060   
1   -6.267057          6        2        2009      2010  ...        15.255135   
2   -6.267057          3        2        2009      2010  ...        15.265584   
3   -6.267057         15        2        2007      2008  ...        15.273393   
4   -6.267057         15        2        2007      2008  ...        15.273435   

  precip_proxy  log_area  sampling_intensity  depth_category  coastal_proxy  \
0  1463

In [32]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd


def run_regularized_models(X_train, X_test, y_train, y_test, label="Target"):
    results = []

    # Ridge
    ridge = Ridge()
    ridge_param_grid = {"alpha": [0.01, 0.1, 1, 10, 100]}
    ridge_cv = GridSearchCV(ridge, ridge_param_grid, cv=5, scoring="r2")
    ridge_cv.fit(X_train, y_train)

    ridge_pred = ridge_cv.predict(X_test)
    results.append({
        "Target": label,
        "Model": "Ridge",
        "Best_Params": ridge_cv.best_params_,
        "R2": r2_score(y_test, ridge_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, ridge_pred))
    })

    # Lasso
    lasso = Lasso(max_iter=100000)
    lasso_param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10]}
    lasso_cv = GridSearchCV(lasso, lasso_param_grid, cv=5, scoring="r2")
    lasso_cv.fit(X_train, y_train)

    lasso_pred = lasso_cv.predict(X_test)
    results.append({
        "Target": label,
        "Model": "Lasso",
        "Best_Params": lasso_cv.best_params_,
        "R2": r2_score(y_test, lasso_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, lasso_pred))
    })

    # ElasticNet
    elastic = ElasticNet(max_iter=100000)
    elastic_param_grid = {
        "alpha": [0.001, 0.01, 0.1, 1],
        "l1_ratio": [0.2, 0.5, 0.8]
    }
    elastic_cv = GridSearchCV(elastic, elastic_param_grid, cv=5, scoring="r2")
    elastic_cv.fit(X_train, y_train)

    elastic_pred = elastic_cv.predict(X_test)
    results.append({
        "Target": label,
        "Model": "ElasticNet",
        "Best_Params": elastic_cv.best_params_,
        "R2": r2_score(y_test, elastic_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, elastic_pred))
    })

    print(f"\n--- Results for {label} ---")
    print("Best alpha params and test performance:")
    for row in results:
        print(f"{row['Model']}:")
        print("  Best params:", row["Best_Params"])
        print("  R²:", round(row["R2"], 4))
        print("  RMSE:", round(row["RMSE"], 4))

    return pd.DataFrame(results), ridge_cv, lasso_cv, elastic_cv

    # Raw species richness
results_richness, ridge_cv_rich, lasso_cv_rich, elastic_cv_rich = run_regularized_models(
    X_rich_train_scaled,
    X_rich_test_scaled,
    y_rich_train,
    y_rich_test,
    label="species_richness"
)

results_density, ridge_cv_density, lasso_cv_density, elastic_cv_density = run_regularized_models(
    X_density_train_scaled,
    X_density_test_scaled,
    y_density_train,
    y_density_test,
    label="log(richness/area)"
)


--- Results for species_richness ---
Best alpha params and test performance:
Ridge:
  Best params: {'alpha': 1}
  R²: 0.439
  RMSE: 20.5368
Lasso:
  Best params: {'alpha': 0.1}
  R²: 0.4423
  RMSE: 20.4758
ElasticNet:
  Best params: {'alpha': 0.01, 'l1_ratio': 0.8}
  R²: 0.4395
  RMSE: 20.527

--- Results for log(richness/area) ---
Best alpha params and test performance:
Ridge:
  Best params: {'alpha': 0.1}
  R²: 0.7468
  RMSE: 0.5914
Lasso:
  Best params: {'alpha': 0.001}
  R²: 0.7449
  RMSE: 0.5937
ElasticNet:
  Best params: {'alpha': 0.001, 'l1_ratio': 0.8}
  R²: 0.7447
  RMSE: 0.5939


In [34]:
coef_rich = pd.DataFrame({
    "Feature": X_rich_train.columns,
    "Ridge": ridge_cv_rich.best_estimator_.coef_,
    "Lasso": lasso_cv_rich.best_estimator_.coef_,
    "ElasticNet": elastic_cv_rich.best_estimator_.coef_
})

print("\nTop predictors for species_richness:")
print(coef_rich.sort_values(by="Lasso", key=np.abs, ascending=False).head(20))


coef_density = pd.DataFrame({
    "Feature": X_density_train.columns,
    "Ridge": ridge_cv_density.best_estimator_.coef_,
    "Lasso": lasso_cv_density.best_estimator_.coef_,
    "ElasticNet": elastic_cv_density.best_estimator_.coef_
})

print("\nTop predictors for log(richness/area):")
print(coef_density.sort_values(by="Lasso", key=np.abs, ascending=False).head(20))


Top predictors for species_richness:
                       Feature      Ridge         Lasso  ElasticNet
22            latitude_squared  20.567983  3.132599e+01   20.050799
25  climate_Temperate/Tropical  36.268225  3.085531e+01   32.872767
5                      n_years  17.213537  2.159089e+01   18.098379
0                     LATITUDE -22.761378 -1.784439e+01  -18.274906
8                   AREA_SQ_KM -20.129593 -1.690165e+01  -19.078914
1                    LONGITUDE  16.511501  1.553404e+01   15.967704
17                    log_area  13.587366  1.058022e+01   12.665122
20                temp_squared  21.469849  9.773404e+00   18.255918
24           climate_Temperate  -6.090737 -7.022563e+00   -7.520769
3                   mean_depth  -7.029775 -6.707580e+00   -6.752942
4                    n_samples   4.133862  4.307338e+00    4.236077
2              total_abundance   4.207062  4.180259e+00    4.210512
6                   year_start -11.414024 -4.044492e+00   -9.392649
11        